<a href="https://colab.research.google.com/github/acoiman/pdt/blob/main/rr_respiratory_mortality/notebooks/04.RR_Respiratory_Mortality_Spatial_Autocorrelation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌐 Global Spatial Autocorrelation Analysis of Predicted Asthma Mortality Rate and Relative Risk of Respiratory Mortality in Argentina, 2022

In this notebook, we evaluate the global spatial autocorrelation of the predicted asthma mortality rate and the RR of respiratory mortality across departments in Argentina for 2022.

Spatial autocorrelation measures the degree to which geographically proximate spatial units exhibit similar attribute values. If departments with high (or low) predicted mortality rates tend to cluster geographically, the data are said to exhibit positive spatial autocorrelation. Conversely, if neighboring departments are dissimilar, the pattern reflects negative spatial autocorrelation.

To assess this, we will:

1. Construct and compare alternative spatial weight matrices (e.g., contiguity-based and distance-based).

2. Select the most appropriate spatial weights structure.

3. Compute Global Moran’s I for:

  * Predicted asthma mortality rate (CA_2022_PRED)

  * RR of respiratory mortality (lower, central and upper bounds)

The analysis is conducted at the department level, using polygon geometries.

## 🤖 Load libraries

The libraries required for the analysis will be loaded

In [ ]:
# dataframe libraries
import pandas as pd

# Import NumPy for numerical and mathematical operations
import numpy as np

# geospatial libraries
import geopandas as gpd

# geostatistical libraries
import pysal
import mapclassify
from libpysal.weights import Queen, KNN
from pysal.lib import weights
import esda
from esda.moran import Moran
from esda.moran import Moran_Local
import libpysal

# plot libraries
import matplotlib.pyplot as plt

# other libraries
import os
from itables import init_notebook_mode, show

In [ ]:
# change directory to work folder (at the begining docker container enter into /home/jovyan/)
%cd work

In [ ]:
# Set the PROJ_LIB path
os.environ['PROJ_LIB'] = "/opt/conda/envs/gds/share/proj"

##🗂 Load and prepare the dataset

In [ ]:
# open gdf data
gdf = gpd.read_file("pdt/rr_respiratory_mortality/data/gpkg/gdf_rrrm.gpkg")

In [ ]:
# visualize geodataframe
init_notebook_mode(all_interactive=True)
gdf

In [ ]:
# visualize column names
gdf.columns

In [ ]:
# select 'IDDPTO', 'AMR_PRD_2022' 'RRRM_2022_L' 'RRRM_2022_C', 'RRRM_2022_U', 'geometry' columns
gdf_ca_rrrm = gdf[[ 'IDDPTO', 'AMR_PRED_2022', 'RRRM_2022_L', 'RRRM_2022_C', 'RRRM_2022_U', 'geometry']]

In [ ]:
# visualize df
init_notebook_mode(all_interactive=True)
gdf_ca_rrrm

In [ ]:
# original length of data
len(gdf_ca_rrrm)

In [ ]:
# drop all rows containing NaN for all rows
gdf_ca_rrrm = gdf_ca_rrrm.dropna().reset_index(drop=True)

In [ ]:
# length of data after dropping NaN
len(gdf_ca_rrrm)

In [ ]:
# visualize gdf
init_notebook_mode(all_interactive=True)
gdf_ca_rrrm

## 🔎 Evaluation and Selection of the Spatial Weights Matrix

Spatial autocorrelation statistics require a spatial weights matrix (W), which encodes the spatial structure of the data.


For polygon data (departments), the most appropriate  spatial matrix are:

1. Contiguity-Based Weights. We define neighbors based on shared boundaries.

    * Rook contiguity → shared borders only

    * Queen contiguity → shared borders OR shared vertices
    
    Contiguity-Based Weights area appropriate for areal units, reflects administrative adjacency, there is no arbitrary distance threshold. However, departments with irregular shapes may have few neighbors

2. Distance-Based Weights. We define neighbors within a given threshold distance.

    * Fixed distance band

    * k-nearest neighbors (kNN)

    Distance-Based Weights ensures minimum connectivity, and is useful when polygons vary greatly in size. However, it requires selecting arbitrary threshold, and may connect geographically unrealistic units

Given that our spatial units are administrative departments and we aim to assess epidemiological spillover patterns, Queen contiguity appears to be the most appropriate initial specification. However, we will construct and evaluate Queen contiguity weights and compare their performance with K-Nearest Neighbors (KNN) weights.

In [ ]:
# project gdf to 5347
gdf_pj = gdf_ca_rrrm.to_crs(5347)

In [ ]:
# visualize gdf_pj (POSGAR 2007 / Argentina 5)
init_notebook_mode(all_interactive=True)
gdf_pj

In [ ]:
# Queen contiguity
w_queen = weights.contiguity.Queen.from_dataframe(gdf_pj, use_index=True)

In [ ]:
# K-NN with five nearest neighbors
w_knn5 = weights.KNN.from_dataframe(gdf_pj, k=5)

In [ ]:
# select all NaN rows for gdf
gdf_nan = gdf[gdf.isna().any(axis=1)]

In [ ]:
# from gdf_nan select IDDPTO and geometry
gdf_nan = gdf_nan[['IDDPTO', 'geometry']]

In [ ]:
# reproject gdf_nan
gdf_nan_pj = gdf_nan.to_crs(5347)

In [ ]:
# Set up figure and axis
f, axs = plt.subplots(1, 2, figsize=(8, 12))

# Queen contiguity matrix
ax = axs[0]
gdf_pj.plot(ax=ax)
w_queen.plot(
    gdf_pj,
    edge_kws=dict(linewidth=1, color="orangered"),
    node_kws=dict(marker="*"),
    ax=ax,
)
# NaN departments, where asthma mortality did not ocurred
gdf_nan_pj.plot(ax=ax, color="none", edgecolor="black", linewidth=0.15)
ax.set_axis_off()
ax.set_title("Queen contiguity matrix")

# KNN
ax = axs[1]
gdf_pj.plot(ax=ax)
w_knn5.plot(
    gdf_pj,
    edge_kws=dict(linewidth=1, color="orangered"),
    node_kws=dict(marker="*"),
    ax=ax,
)
# NaN departments, where asthma mortality did not ocurred
gdf_nan_pj.plot(ax=ax, color="none", edgecolor="black", linewidth=0.15)
ax.set_axis_off()
ax.set_title("K-nearest neighbor weights (k=5)")

f.tight_layout()
plt.show()

In [ ]:
# Check for isolated departments (no neighbors)
print("Number of islands (Queen):", len(w_queen.islands))

In [ ]:
# Check for isolated departments (no neighbors)
print("Number of islands (KNN-5):", len(w_knn5.islands))

In [ ]:
# visualize percentage of non-zero weights (connections)
pct_nonzero_data = {
    'Weight Type': ['Queen', 'KNN (k=5)'],
    'pct_nonzero': [
        w_queen.pct_nonzero,
        w_knn5.pct_nonzero,
    ]
}
df_weights = pd.DataFrame(pct_nonzero_data)

In [ ]:
# Display the DataFrame
init_notebook_mode(all_interactive=False)
df_weights

The comparison between Queen contiguity and KNN (k=5) highlights marked differences in spatial connectivity across Argentine departments. The Queen criterion generates 16 islands—spatial units without shared borders. This produces a disconnected spatial graph and a relatively sparse weights matrix (1.49% nonzero elements), emphasizing strictly local, border-based interactions. In contrast, the KNN (k=5) specification eliminates islands entirely by construction, ensuring that each unit is connected to its five nearest neighbors. This results in a fully connected spatial graph with higher density (2.94% nonzero elements), though it introduces links that may connect geographically unrealistic units.

From a methodological standpoint, these structural differences have important implications for spatial  analysis. Queen contiguity preserves true geographic connections and is appropriate when modeling processes rely on shared borders; however, disconnected components may cause inference. KNN (k=5), while less geographically strict, provides statistical stability and guarantees matrix connectivity, which is advantageous for computing global spatial autocorrelation statistics and spatial modeling. Thus, in this context we will compute the Global Moran’s I comparison under both schemes.

## 💻 Computing Global Moran’s I

### Global Moran’s I for Predicted Asthma Mortality Rate

At this stage, we formally test whether the predicted asthma mortality rate in 2022  exhibits global spatial autocorrelation across Argentine departments.

Specifically, we test:

* $H_0$​:Spatial randomness (no spatial autocorrelation)
* $H_1$​:Spatial clustering (positive or negative autocorrelation)

We compute Global Moran’s I under both:

* Queen contiguity matrix

* K-nearest neighbor weights (k=5)

#### Queen contiguity matrix Moran’s I

In [ ]:
# Extract predicted asthma mortality rate as numpy array
# Flatten ensures 1-dimensional structure required by esda
y = gdf_pj["AMR_PRED_2022"].values.flatten()

In [ ]:
# transforms raw weights into row-standardized
w_queen.transform = 'R'

In [ ]:
# Fix the random number generator seed
np.random.seed(42)

# Compute Moran's I using 999 random permutations
moran_queen = Moran(y, w_queen, permutations=999)

# Print results
print("Global Moran’s I (Queen)")
print("---------------------------")
print("Moran's I:", moran_queen.I)
print("Expected I:", moran_queen.EI)
print("p-value (permutation):", moran_queen.p_sim)
print("Z-score:", moran_queen.z_sim)

The Global Moran’s I under Queen contiguity matrix (I ≈ 0.27) indicates moderate positive spatial autocorrelation in the predicted asthma mortality rate  across Argentine departments. The observed value is  higher than the expected value under spatial randomness (E[I] ≈ −0.005). The permutation-based p-value (0.002) and high standardized z-score (≈ 3.77) indicate that this clustering pattern is  statistically significant. Therefore, predicted asthma mortality rates are not randomly distributed but exhibit  high–high and low–low spatial clustering, consistent with geographically structured  patterns.

#### K-nearest neighbor weights (k=5) Moran’s I

In [ ]:
# Fix the random number generator seed
np.random.seed(42)

# Compute Moran’s I with 999 permutations
moran_knn = Moran(y, w_knn5, permutations=999)

In [ ]:
# Print results
print("\nGlobal Moran’s I (KNN-5)")
print("---------------------------")
print("Moran's I:", moran_knn.I)
print("Expected I:", moran_knn.EI)
print("p-value (permutation):", moran_knn.p_sim)
print("Z-score:", moran_knn.z_sim)

Under the KNN (k=5) specification, the Global Moran’s I (I ≈ 0.223) also indicates positive spatial autocorrelation in the predicted asthma mortality rate. Although the magnitude is slightly lower than under Queen contiguity (≈ 0.272), it remains substantially above the expected value under spatial randomness (E[I] ≈ −0.005), confirming systematic spatial structure. The permutation-based p-value (0.001) and  high z-score (≈5.36) indicate  statistical significance, with even greater standardized departure from randomness than under Queen weights.

#### Global Moran’s I for RR of respiratory mortality (lower, central and upper bounds)

In this section, we will evaluate whether the RR of respiratory mortality  (lower, central, and upper bounds) exhibit global spatial autocorrelation  across Argentine departments.

Since these variables represent uncertainty bounds of relative risk, assessing spatial autocorrelation allows us to determine whether:

* Spatial clustering is robust across uncertainty intervals,

* Or clustering is sensitive to the bound specification.

If all three measures exhibit similar spatial dependence, this strengthens the spatial robustness of the relative risk estimates.

##### Global Moran’s I for  RR of Respiratory Mortality (Lower Bound)

In [ ]:
# Convert column to 1D numpy array
y_L = gdf_pj["RRRM_2022_L"].values.flatten()

In [ ]:
# Fix the random number generator seed
np.random.seed(42)

# Moran’s I — KNN-5 (Lower Bound)
moran_L_knn = Moran(y_L, w_knn5, permutations=999)

print("Global Moran’s I for  RR of Respiratory Mortality (Lower Bound) — KNN-5")
print("-------------------------")
print("Moran's I:", moran_L_knn.I)
print("Expected I:", moran_L_knn.EI)
print("p-value:", moran_L_knn.p_sim)
print("Z-score:", moran_L_knn.z_sim)

##### Global Moran’s I for  RR of Respiratory Mortality (Central Bound)

In [ ]:
# Extract Central Attributable Case Rate
y_C = gdf_pj["RRRM_2022_C"].values.flatten()

In [ ]:
# Fix the random number generator seed
np.random.seed(42)

# Moran’s I — Queen (Central)
moran_C_queen = Moran(y_C, w_queen, permutations=999)

print("Global Moran’s I for RR of Respiratory Mortality (central Bound)— Queen")
print("-------------------------")
print("Moran's I:", moran_C_queen.I)
print("Expected I:", moran_C_queen.EI)
print("p-value:", moran_C_queen.p_sim)
print("Z-score:", moran_C_queen.z_sim)

In [ ]:
# Fix the random number generator seed
np.random.seed(42)

# Moran’s I — KNN-5 (Central)
moran_C_knn = Moran(y_C, w_knn5, permutations=999)

print("Global Moran’s I for RR of Respiratory Mortality (central Bound) — KNN-5")
print("-------------------------")
print("Moran's I:", moran_C_knn.I)
print("Expected I:", moran_C_knn.EI)
print("p-value:", moran_C_knn.p_sim)
print("Z-score:", moran_C_knn.z_sim)

##### Global Moran’s I for  RR of Respiratory Mortality (Upper Bound)

In [ ]:
# Extract Upper Attributable Case Rate
y_U = gdf_pj["RRRM_2022_U"].values.flatten()

In [ ]:
# Fix the random number generator seed
np.random.seed(42)

# Moran’s I — Queen (Upper)
moran_U_queen = Moran(y_U, w_queen, permutations=999)

print("Global Moran’s I for RR of Respiratory Mortality (Upper Bound) — Queen")
print("-------------------------")
print("Moran's I:", moran_U_queen.I)
print("Expected I:", moran_U_queen.EI)
print("p-value:", moran_U_queen.p_sim)
print("Z-score:", moran_U_queen.z_sim)

In [ ]:
# Fix the random number generator seed
np.random.seed(42)

# Moran’s I — KNN-5 (Upper)
moran_U_knn = Moran(y_U, w_knn5, permutations=999)

print("Global Moran’s I for RR of Respiratory Mortality (Upper Bound) — KNN-5")
print("-------------------------")
print("Moran's I:", moran_U_knn.I)
print("Expected I:", moran_U_knn.EI)
print("p-value:", moran_U_knn.p_sim)
print("Z-score:", moran_U_knn.z_sim)

##### Resume of global Moran’s I for  RR of respiratory mortality (Lower, Central and Upper Bounds)

In [ ]:
moran_results = [
    {
        "Variable": "Respiratory Mortality RR Lower Bound",
        "Weight Type": "Queen",
        "Moran's I": f"{moran_L_queen.I:.2f}",
        "p-value": f"{moran_L_queen.p_sim:.3f}",
        "Z-score": f"{moran_L_queen.z_sim:.2f}",
    },
    {
        "Variable": "Respiratory Mortality RR Lower Bound",
        "Weight Type": "KNN-5",
        "Moran's I": f"{moran_L_knn.I:.2f}",
        "p-value": f"{moran_L_knn.p_sim:.3f}",
        "Z-score": f"{moran_L_knn.z_sim:.2f}",
    },
    {
        "Variable": "Respiratory Mortality RR Central Bound",
        "Weight Type": "Queen",
        "Moran's I": f"{moran_C_queen.I:.2f}",
        "p-value": f"{moran_C_queen.p_sim:.3f}",
        "Z-score": f"{moran_C_queen.z_sim:.2f}",
    },
    {
        "Variable": "Respiratory Mortality RR Central Bound",
        "Weight Type": "KNN-5",
        "Moran's I": f"{moran_C_knn.I:.2f}",
        "p-value": f"{moran_C_knn.p_sim:.3f}",
        "Z-score": f"{moran_C_knn.z_sim:.2f}",
    },
    {
        "Variable": "Respiratory Mortality RR Upper Bound",
        "Weight Type": "Queen",
        "Moran's I": f"{moran_U_queen.I:.2f}",
        "p-value": f"{moran_U_queen.p_sim:.3f}",
        "Z-score": f"{moran_U_queen.z_sim:.2f}",
    },
    {
        "Variable": "Respiratory Mortality RR Upper Bound",
        "Weight Type": "KNN-5",
        "Moran's I": f"{moran_U_knn.I:.2f}",
        "p-value": f"{moran_U_knn.p_sim:.3f}",
        "Z-score": f"{moran_U_knn.z_sim:.2f}",
    },
]

df_moran_summary = pd.DataFrame(moran_results)

In [ ]:
# visualize the table
init_notebook_mode(all_interactive=False)
df_moran_summary

Across all three bounds of respiratory mortality RR (lower, central, and upper), the results indicate statistically significant positive global spatial autocorrelation under both Queen and KNN-5 weights, with the Queen matrix performing slightly better. Moran’s I values range from approximately 0.53 to 0.60, indicating moderate to high spatial clustering of RR across departments. This pattern is consistent across uncertainty bounds, suggesting that spatial autocorrelation is not sensitive to interval specification. Overall, these results indicate that respiratory mortality RR exhibits a structured geographic clustering pattern.

## ✔️ Conclusion


From a modeling perspective—given that we will use SAR (Spatial Autoregressive Model) and/or SEM (Spatial Error Model) to assess the spatial association between the predicted asthma mortality rate and the respiratory mortality RR—the Queen contiguity weight matrix appears to be the more robust specification. Although KNN produces higher z-scores (i.e., stronger evidence against spatial randomness), the Queen matrix yields larger Moran’s I values, indicating stronger positive spatial autocorrelation.

Moreover, KNN can generate unrealistic spatial connections that may bias the maximum likelihood estimation of SAR/SEM models. Therefore, for the spatial regression analysis of the association between the predicted asthma mortality rate and the respiratory mortality RR, the Queen contiguity matrix should be used as the primary specification, with KNN retained for sensitivity analysis to assess distance-based robustness.